# HPO no `guato_ba`: Optuna (o que eu usei) + Neptune (como integrar)

**Leitura honesta primeiro:** no estudo `guato_ba` a otimização de hiperparâmetros (HPO) foi feita com **Optuna** (roda local, sem conta). O **Neptune** eu deixei *preparado como rastreador opcional*, mas **não ativei** porque ele exige um token de conta (`NEPTUNE_API_TOKEN`) que não foi configurado.

Este notebook ensina **os dois**:
1. **Optuna** — como eu de fato otimizei (RF, XGBoost, MLP, SVM e as CNNs), com os **resultados reais** carregados do disco.
2. **Neptune** — o que é e **como adicionar** rastreamento de experimentos (inclusive integrado ao Optuna), que é o que você quer aprender.

> Rode com o kernel do env `geospatial` (onde estão optuna, neptune, sklearn, xgboost).

## Os dois papéis — não competem, se complementam

| | **Optuna** | **Neptune** |
|---|---|---|
| Papel | **Otimizador** de hiperparâmetros | **Rastreador** de experimentos (nuvem) |
| Faz o quê | *escolhe* quais hiperparâmetros testar e *busca* o melhor | *registra* parâmetros, métricas, gráficos e artefatos de cada run |
| Precisa de internet/conta? | Não (roda local, salva em SQLite) | Sim (token + projeto em neptune.ai) |
| No guato_ba | **usado** (motor do HPO) | preparado, **opcional** |

Padrão comum (e o que o guato_ba já suporta): **Optuna otimiza + Neptune registra cada tentativa**.

In [ ]:
import optuna, sys
print("optuna", optuna.__version__)
try:
    import neptune; print("neptune", neptune.__version__)
except Exception as e:
    print("neptune indisponível:", e)
import os
print("NEPTUNE_API_TOKEN definido?", bool(os.environ.get("NEPTUNE_API_TOKEN")))

---
## Parte 1 — Optuna: o conceito

Três peças:
- **`study`**: o experimento de busca (maximizar ou minimizar uma métrica).
- **`objective(trial)`**: função que recebe um `trial`, *sugere* hiperparâmetros (`trial.suggest_*`), treina um modelo e **retorna a métrica** (aqui, IoU de validação).
- **`study.optimize(objective, n_trials=...)`**: roda N tentativas; o Optuna usa os resultados passados para escolher os próximos hiperparâmetros (amostragem TPE, bem melhor que grade/aleatório).

Exemplo mínimo e executável (Random Forest em dados sintéticos), **espelhando exatamente** a estrutura do `guato_ba`:

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification

# dados de brinquedo (no guato_ba são pixels: 8 bandas t1/t2 + dNBR/dNBRSWIR/dMIRBI = 11 features)
X, y = make_classification(n_samples=20000, n_features=11, n_informative=6, weights=[0.5,0.5], random_state=42)
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.3, random_state=42)

def iou(clf):
    clf.fit(Xtr, ytr); p = clf.predict(Xva)
    TP=int(((p==1)&(yva==1)).sum()); FP=int(((p==1)&(yva==0)).sum()); FN=int(((p==0)&(yva==1)).sum())
    return TP/(TP+FP+FN) if (TP+FP+FN) else 0.0   # IoU = CSI (métrica do artigo)

def objective(trial):
    # ESTE é o 'espaço de busca' — igual ao espaco_optuna_ml('rf', trial) do guato_ba
    hp = dict(
        n_estimators = trial.suggest_int("n_estimators", 120, 350, step=50),
        max_depth    = trial.suggest_int("max_depth", 10, 28),
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 8),
        max_features = trial.suggest_categorical("max_features", ["sqrt","log2"]),
    )
    return iou(RandomForestClassifier(n_jobs=4, class_weight="balanced", random_state=42, **hp))

study = optuna.create_study(direction="maximize")   # queremos MAX IoU
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=12)
print("melhor IoU:", round(study.best_value,4))
print("melhores hiperparâmetros:", study.best_params)

### O que acabou de acontecer
- O Optuna rodou 12 `trials`. Em cada uma, `suggest_*` propôs um conjunto de hiperparâmetros, treinamos o RF e devolvemos o IoU.
- `direction="maximize"` → ele busca o **maior** IoU.
- `study.best_params` é a melhor configuração encontrada; `study.best_value`, a melhor métrica.

No guato_ba, isso vive em `main.py::hpo_train_ml()` e `redes.py::espaco_optuna_ml()`. Depois do HPO, o modelo final é **retreinado com `best_params`** em treino+val e salvo (`machine_learning/<modelo>/<modelo>.pkl`).

### Persistência: estudo em SQLite (reentrante)
No guato_ba eu salvo cada estudo num banco SQLite (`hpo/ml_<modelo>.db`) com `storage=` e `load_if_exists=True` — assim o HPO pode ser **retomado** e os resultados ficam **auditáveis**. Vamos carregar o estudo **real** do Random Forest do guato_ba:

In [ ]:
import optuna, os
DB = "hpo/ml_rf.db"   # rode este notebook a partir de guato_ba/notebooks com cwd em guato_ba, ou ajuste o caminho
path = DB if os.path.exists(DB) else ("../"+DB if os.path.exists("../"+DB) else None)
if path:
    st = optuna.load_study(study_name="ml_rf", storage=f"sqlite:///{path}")
    print("trials:", len(st.trials), "| melhor IoU:", round(st.best_value,4))
    print("best_params:", st.best_params)
    df = st.trials_dataframe()[["number","value","params_n_estimators","params_max_depth","params_min_samples_leaf","params_max_features","state"]]
    display(df.sort_values("value", ascending=False).head(10))
else:
    print("estudo real não encontrado — rode a partir da pasta guato_ba (ou ajuste DB)")

### Importância dos hiperparâmetros
O Optuna estima quanto cada hiperparâmetro influenciou a métrica (fANOVA). Útil para saber onde focar.

In [ ]:
import optuna
if path:
    try:
        imp = optuna.importance.get_param_importances(st)
        for k,v in imp.items(): print(f"  {k:22s} {v:.3f}")
    except Exception as e:
        print("não deu p/ calcular importância:", e)
    # gráficos (precisa de matplotlib)
    try:
        import matplotlib.pyplot as plt
        optuna.visualization.matplotlib.plot_optimization_history(st); plt.tight_layout(); plt.show()
        optuna.visualization.matplotlib.plot_param_importances(st); plt.tight_layout(); plt.show()
    except Exception as e:
        print("viz:", e)

---
## Parte 1b — Optuna para as CNNs (deep learning)

Para os modelos de deep learning (unet, att_unet, deeplabv3plus, ...), o HPO é **curto** (poucas épocas por trial, com *pruning*) só para achar bons hiperparâmetros; depois treino o modelo **final** 1000 épocas com early-stop usando a melhor config.

O espaço de busca das CNNs (em `redes.py::espaco_optuna_dl`) tuna **learning rate, dropout e os pesos da loss** (Focal Tversky `tv_b` = peso dos falsos negativos, e `w_bce`):

In [ ]:
# espaço de busca real das CNNs no guato_ba
def espaco_optuna_dl(trial):
    return dict(
        lr      = trial.suggest_float("lr", 5e-5, 8e-4, log=True),   # learning rate (log-uniforme)
        dropout = trial.suggest_float("dropout", 0.0, 0.4),
        tv_b    = trial.suggest_float("tv_b", 0.45, 0.8),            # peso FN na Focal Tversky (controla omissão/comissão)
        w_bce   = trial.suggest_float("w_bce", 0.25, 1.5),
    )
# demonstração: quais valores o Optuna sugeriria nas primeiras trials
s = optuna.create_study(direction="maximize")
for _ in range(5):
    t = s.ask(); print(espaco_optuna_dl(t)); s.tell(t, 0.0)

No guato_ba (`main.py::hpo_dl`), o objetivo treina a CNN por ~40 épocas e retorna o **IoU de validação**; uso `MedianPruner` para cortar trials ruins cedo. Fluxo:

```
hpo_dl(modelo) -> Optuna (10 trials x ~40 ep, pruning) -> hpo_best.json
train-dl(modelo) -> treino final 1000 ep (early-stop) com a melhor config
```

Rodar na linha de comando:
```bash
python main.py hpo-dl  --modelo unet --trials 10   # busca
python main.py train-dl --modelo unet              # treino final c/ a melhor config
```

---
## Parte 2 — Neptune: rastreando os experimentos

O **Neptune** (neptune.ai) é um *tracker* na nuvem: cada `run` registra hiperparâmetros, métricas ao longo do tempo, gráficos e arquivos (checkpoints, imagens). Serve para **comparar experimentos** num dashboard e não perder o histórico.

**Setup (uma vez):**
1. Criar conta em neptune.ai e um **projeto** (ex.: `seu_usuario/guato_ba`).
2. Pegar o **API token** e exportar no ambiente (no terminal do Jupyter):
   ```bash
   ! export NEPTUNE_API_TOKEN="eyJ...seu_token..."
   ! export NEPTUNE_PROJECT="seu_usuario/guato_ba"
   ```
   (ou passar `api_token=` e `project=` direto no código).

O código abaixo é **guardado**: só executa de verdade se houver token; senão, mostra o que faria.

In [ ]:
import os
TOKEN = os.environ.get("NEPTUNE_API_TOKEN")
PROJECT = os.environ.get("NEPTUNE_PROJECT", "seu_usuario/guato_ba")
if TOKEN:
    import neptune
    run = neptune.init_run(project=PROJECT, api_token=TOKEN, name="exemplo-rf")
    run["params"] = {"modelo":"rf","n_estimators":270,"max_depth":21}   # registra hiperparâmetros
    for epoch, iou_val in enumerate([0.90,0.93,0.95,0.96]):             # métricas ao longo do tempo
        run["val/iou"].append(iou_val)
    run["val/iou_best"] = 0.96
    # run["artifacts/modelo"].upload("machine_learning/rf/rf.pkl")      # (opcional) subir o checkpoint
    run.stop()
    print("run enviado ao Neptune:", PROJECT)
else:
    print("Sem NEPTUNE_API_TOKEN — exemplo NÃO enviado.")
    print("O que o código faria: criar um run, logar params em run['params'],")
    print("métricas com run['val/iou'].append(...), e artefatos com run[...].upload(...).")

---
## Parte 3 — Optuna **+** Neptune juntos (o padrão recomendado)

O pacote `neptune-optuna` liga os dois: um **callback** registra automaticamente cada trial do Optuna no Neptune (valores, parâmetros, melhor trial, gráficos de importância/história). Assim você **otimiza com Optuna e acompanha no dashboard do Neptune**.

In [ ]:
import os, optuna
TOKEN = os.environ.get("NEPTUNE_API_TOKEN")
if TOKEN:
    import neptune
    import neptune.integrations.optuna as nept_optuna
    run = neptune.init_run(project=os.environ.get("NEPTUNE_PROJECT","seu_usuario/guato_ba"),
                           api_token=TOKEN, name="optuna-rf")
    neptune_cb = nept_optuna.NeptuneCallback(run)          # <- o elo Optuna->Neptune
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=12, callbacks=[neptune_cb])   # 'objective' da Parte 1
    print("melhor:", study.best_params, round(study.best_value,4))
    run.stop()
else:
    print("Sem token: com NEPTUNE_API_TOKEN, bastaria adicionar callbacks=[NeptuneCallback(run)]")
    print("ao study.optimize(...) que você já viu na Parte 1 — nada mais muda no HPO.")

### Como eu deixei isso pronto no guato_ba
- `neptune` e `neptune-optuna` já estão **instalados**.
- O HPO (Optuna) já roda e salva tudo em `hpo/*.db` + `*/hpo_best.json`.
- Para **ativar o Neptune**, basta exportar `NEPTUNE_API_TOKEN`/`NEPTUNE_PROJECT` e adicionar `callbacks=[NeptuneCallback(run)]` nas chamadas `study.optimize(...)` de `main.py` (`hpo_train_ml` e `hpo_dl`).

## Resumo
- **Optuna = quem busca** os hiperparâmetros (foi o que rodou de verdade; resultados em `hpo/*.db` e `*/hpo_best.json`).
- **Neptune = quem registra/compara** os experimentos num dashboard na nuvem (opcional, precisa de token).
- Juntos: `study.optimize(objective, callbacks=[NeptuneCallback(run)])`.

Resultados reais do meu HPO (ML, IoU de validação balanceada): rf 0,992 · xgboost 0,992 · mlp 0,992 · svm 0,990. Para as CNNs, a melhor no teste 2025 (OOT) foi a **unet**.